In [17]:
# Installing updated libraries for sequence labeling and evaluation
!pip install transformers[torch] datasets evaluate seqeval

In [19]:
from datasets import load_dataset
from transformers import AutoTokenizer

# 1. Loading a script-free version of CoNLL-2003 to ensure compatibility
raw_datasets = load_dataset("lhoestq/conll2003")

# 2. Defining standard NER labels manually for stability
label_list = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

# 3. Initializing DistilBERT tokenizer (Fast and Efficient)
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Special tokens (CLS, SEP)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100) # Sub-tokens of the same word
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# 4. Map the preprocessing function across the dataset
tokenized_datasets = raw_datasets.map(tokenize_and_align_labels, batched=True)

In [21]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import numpy as np
import evaluate

# 1. Loading DistilBERT for Token Classification
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(label_list))

# 2. Loading the evaluation metric
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 3. Optimized Training Arguments for speed and compliance
args = TrainingArguments(
    "ner-distilbert-assignment",
    eval_strategy="epoch",    # Modern strategy name
    save_strategy="epoch",
    learning_rate=2e-5,       # Mandatory Learning Rate
    per_device_train_batch_size=16,
    num_train_epochs=3,       # Mandatory Epochs
    weight_decay=0.01,
    load_best_model_at_end=True
)

# 4. Initialize the Trainer with the corrected tokenizer/processing_class argument
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"].shuffle(seed=42).select(range(1500)),
    eval_dataset=tokenized_datasets["validation"].select(range(300)),
    data_collator=DataCollatorForTokenClassification(tokenizer),
    processing_class=tokenizer, # FIXED: Use processing_class instead of tokenizer
    compute_metrics=compute_metrics
)

print("--- Starting Training on NER Task ---")
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


--- Starting Training on NER Task ---


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.369122,0.571678,0.612360,0.591320,0.918362
2,No log,0.206233,0.738977,0.784644,0.761126,0.952736
3,No log,0.171145,0.794643,0.833333,0.813528,0.963334


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=282, training_loss=0.30921016179078015, metrics={'train_runtime': 1319.8914, 'train_samples_per_second': 3.409, 'train_steps_per_second': 0.214, 'total_flos': 55965397016592.0, 'train_loss': 0.30921016179078015, 'epoch': 3.0})

In [22]:
from transformers import pipeline

# Creating a high-level pipeline for testing
nlp = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

# Testing on a new sentence
sample_text = "Sushil Panthi is a developer currently working in Nepal."
results = nlp(sample_text)

print("\n--- Inference Results ---")
for entity in results:
    print(f"Entity: {entity['word']} | Label: {entity['entity_group']} | Score: {entity['score']:.4f}")


--- Inference Results ---
Entity: sushil | Label: LABEL_1 | Score: 0.9267
Entity: panthi | Label: LABEL_2 | Score: 0.8892
Entity: is a developer currently working in | Label: LABEL_0 | Score: 0.9899
Entity: nepal | Label: LABEL_5 | Score: 0.8996
Entity: . | Label: LABEL_0 | Score: 0.9905


# **Project Summary & Insights**

### **1. Architecture Selection**
For this token classification task, I utilized **DistilBERT**, a lightweight version of BERT. This was chosen specifically to optimize training speed while retaining over 95% of the performance of the full BERT-base model. It ensures the model is efficient for production-like environments.

### **2. Token Alignment Strategy**
Since Transformers use sub-word tokenization (WordPieces), a single word can be split into multiple tokens. I implemented a robust alignment strategy using `word_ids` to ensure that only the first sub-token of a word carries the label, while the rest are ignored by the loss function (using the `-100` index).

### **3. Evaluation Metrics**
Rather than simple accuracy, the model was evaluated using the **seqeval** framework. This provides a more rigorous assessment by calculating **Precision, Recall, and F1-score** at the entity level, which is critical for sequence labeling tasks like NER and POS tagging.

### **4. Conclusion**
The model successfully identifies entities such as Persons (PER), Organizations (ORG), and Locations (LOC). By using the **AdamW** optimizer and a fine-tuning learning rate of **2e-5**, the system achieved smooth convergence and reliable inference results.